In [20]:
import os
from dotenv import load_dotenv
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage
from typing import List, Dict, Any
import time


load_dotenv()

True

In [2]:
# read all the pdf's inside the folder
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # add source information to metadata
            for doc in documents:
                doc.metadata['source-file'] = pdf_file.name
                doc.metadata['file-type'] = 'pdf'

            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")

        except Exception as e:
            print(f"error {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")


Found 12 PDF files to process

Processing: 01. Bidirection-RNN.pdf
loaded 2 pages

Processing: 01. EncoderDecoderSeq2SEq.pdf
loaded 4 pages

Processing: Attention is all you need.pdf
loaded 15 pages

Processing: Correlation Analysis.pdf
loaded 9 pages

Processing: deepseek.pdf
loaded 86 pages

Processing: Embed Documents Using watsonx’s Embedding Model.pdf
loaded 2 pages

Processing: Exponential distribution.pdf
loaded 6 pages

Processing: Poisson Distribution.pdf
loaded 11 pages

Processing: Reading - Compare Fine-Tuning Using InstructLab with RAG.pdf
loaded 2 pages

Processing: Regression Analysis.pdf
loaded 4 pages

Processing: Test of Hypothesis.pdf
loaded 11 pages

Processing: Theory of Probability.pdf
loaded 12 pages

Total documents loaded: 164


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\01. Bidirection-RNN.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source-file': '01. Bidirection-RNN.pdf', 'file-type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\01. Bidirection-RNN.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2', 'source-file': '01. Bidirection-RNN.pdf', 'file-type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\01. EncoderDecoderSeq2SEq.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source-file': '01. EncoderDecoderSeq2SEq.pdf', 'file-type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\01. EncoderDecoderSeq2SEq.pdf', 'total_pages': 4, 'page': 1, 'page_label': '2', 'source-file': 

In [4]:
# Text splitting get into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split 164 documents into 481 chunks

Example chunk:
Content: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Attention is all you need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source-file': 'Attention is all you need.pdf', 'file-type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Attention is all you need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source-file': 'Attention is all you need.pdf', 'file-type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAi

## Embedding and VetorDB

In [6]:
class Embedding_Manager:
    """Handles document embedding generation using sentence transformer"""

    def __init__(self, model_name:str = 'all-MiniLM-L6-v2'):
        """
        Initialize the embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddigns
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise


    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


embedding_manager = Embedding_Manager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9360.87it/s]


Model loaded successfully. Embedding dimension: 384


In [7]:
# vector store
class Vector_Store:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name:str = "pdf_documents", persist_directory:str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
             collection_name: Name of the ChromaDB collection
             persist_directory: Directory to persist the vector
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self.initialize_store()

    def initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"descriptio": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store")

        # preparing data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['context_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # document content
            documents_text.append(doc.page_content)
            # embedding
            embeddings_list.append(embedding.tolist())

        # add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                documents=documents_text,
                metadatas=metadatas,
            )

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vector_store = Vector_Store()
vector_store


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 1015


In [8]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Attention is all you need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source-file': 'Attention is all you need.pdf', 'file-type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAi

In [9]:
# convert the text to embeddings
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

# store in the vector database
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 481 texts...


Batches: 100%|██████████| 16/16 [00:08<00:00,  1.80it/s]


Generated embeddings with shape: (481, 384)
Adding 481 documents to vector store
Successfully added 481 documents to vector store
Total documents in collection: 1496


## Retriever Pipeline

In [10]:
class RAG_Retriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store:Vector_Store, embedding_manager:Embedding_Manager):
        """
        Initialize the retriever

            Args:
                vector_store: vector store containing document embeddings
                embedding_manager: manager for generating query embeddings

        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query:str, top_k:int = 5, score_threshold:float = 0.0) -> List[Dict[str, Any]]:
        """ 
        Retrieve relevant topics from documents for a query

        Args:
            query: the search query
            top_K: number of top results to return
            score_threshold: minimum similarity score threshold

        Returns:
        List of dictionaries containing retrieved documents and metadata

        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # search in the vector store
        try:
            results = self.vector_store.collection.query(query_embeddings=[query_embedding.tolist()], n_results=top_k)
            # process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'distance': distance,
                            'similarity_score': similarity_score,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever = RAG_Retriever(vector_store, embedding_manager)
rag_retriever

In [11]:
rag_retriever.retrieve("What is Attention is all you need")

Retrieving documents for query: 'What is Attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 166.71it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_9a1dea98_12',
  'content': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3',
  'metadata': {'page': 2,
   'trapped': '/False',
   'keywords': '',
   'file-type': 'pdf',
   'context_length': 216,
   'source-file': 'Attention is all you need.pdf',
   'title': '',
   'source': '..\\data\\pdf\\Attention is all you need.pdf',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'doc_index': 12,
   'page_label': '3',
   'subject': '',
   'total_pages': 15,
   'moddate': '2024-04-10T21:11:43+00:00',
   'author': '',
   'creator': 'LaTeX with hyperref',
   'creationdate': '2024-04-10T21:11:43+00:00',
   'producer': 'pdfTeX-1.40.25'},
  'distance': 0.8600451946258545,
  'similarity_score': 0.1399548053741455,
  'rank': 1},
 {'id': 'doc_cd228d1d_12',
 

## VectorDB to LLM Output

In [12]:
class GroqLLM:
    def __init__(self, model_name:str = "gemma2-9b-it", api_key: str = None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
            
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")

        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")

        self.llm = ChatGroq(
            groq_api_key = self.api_key,
            model = self.model_name,
            temperature = 0.1,
            max_tokens = 1024
        )

        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate(self, query:str, context:str, max_length:int = 500) -> str:
        """
        Generate response using retrieved context
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """

        # create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "questions"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.
            Context:{context}
            Question: {question}
            Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )

        formatted_prompt = prompt_template.format(context=context, question=query)

        try:
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error generating response: {str(e)}"

    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

            Question: {query}

            Answer:"""

        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
        


In [13]:
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [14]:
rag_retriever.retrieve("Attention Mechanism")

Retrieving documents for query: 'Attention Mechanism'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 142.84it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_9a1dea98_12',
  'content': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3',
  'metadata': {'context_length': 216,
   'subject': '',
   'source': '..\\data\\pdf\\Attention is all you need.pdf',
   'source-file': 'Attention is all you need.pdf',
   'total_pages': 15,
   'author': '',
   'trapped': '/False',
   'keywords': '',
   'page_label': '3',
   'creationdate': '2024-04-10T21:11:43+00:00',
   'creator': 'LaTeX with hyperref',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'page': 2,
   'doc_index': 12,
   'moddate': '2024-04-10T21:11:43+00:00',
   'file-type': 'pdf',
   'producer': 'pdfTeX-1.40.25',
   'title': ''},
  'distance': 0.7187539935112,
  'similarity_score': 0.28124600648880005,
  'rank': 1},
 {'id': 'doc_cd228d1d_12',
  '

## Integration Vectordb Context pipeline With LLM output

In [15]:
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [16]:
answer=rag_simple("What is attention mechanism?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is attention mechanism?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 142.84it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


An attention mechanism is a function that maps a query and key-value pairs to an output, computing it as a weighted sum of the values, where all inputs and outputs are vectors.


## Enhanced RAG

In [19]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

result = rag_advanced("Regression Analysis", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Regression Analysis'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 166.71it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Regression analysis is a statistical method used to quantify relationships between variables, provide forecasts and predictions, and describe these relationships using measured values and graphical representations. Its applications include financial forecasting, crime data mining, handwriting recognition, software cost prediction, credit scoring, and healthcare cost prediction.
Sources: [{'source': '..\\data\\pdf\\Regression Analysis.pdf', 'page': 0, 'score': 0.1615898609161377, 'preview': 'Some examples of the related variables are:\ni. Fertilizer used and yield of various plots of land.\nii. Income and expenditure of a class of people. \niii. The price of commodity and amount demanded.\niv. The advertising expenditure and the volume of sales of a product. \nv. The heigh and weight of stud...'}, {'source': '..\\data\\pdf\\Regression Analysis.pdf', 'page': 0, 'score': 0.1615898609161377, 'preview': 'Some examples of the related variables are:\ni. Fertilizer used and yield of va

In [21]:
# Advanced RAG
class Advanced_RAG:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join(doc['content'] for doc in results)
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # streaming
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streamming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()

            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }




adv_rag = Advanced_RAG(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])






Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 166.63it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streamming answer:
Use the following context to answer the question concisely.
Context:
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

3.2 Attention
An attention functi

on can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

Question: what is attention is all you need

Answer:

Final Answer: "Attention is All You Need" refers to a paper that introduced the Transformer model, which relies entirely on attention mechanisms to process input sequences, eliminating the need for recurrent neural networks (RNNs) and convolutional neural networks (CNNs).

Citations:
[1] ..\data\pdf\Attention is all you need.pdf (page 2)
[2] ..\data\pdf\Attention is all you need.pdf (page 2)
[3] ..\data\pdf\Attention is all you need.pdf (page 2)
Summary: The paper "Attention is All You Need" introduced the Transformer model, a new approa